In [ ]:
# Import library yang diperlukan
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras_tuner import RandomSearch
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import Precision, Recall
import tensorflow as tf

In [2]:
# Konfigurasi
DATASET_PATH = r'D:\1_arsip negara\1_arsip_unsr!_on going\Semester 7\1_AICI Batch 7\Tugas Akhir\Final_Project\bisindo'
# Direktori masing-masing
train_dir = os.path.join(DATASET_PATH, 'Train')
val_dir = os.path.join(DATASET_PATH, 'Validation')
test_dir = os.path.join(DATASET_PATH, 'Test')

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 26

In [3]:
# Membuat Train, Validation, dan Test Data Generator
train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.1,
    horizontal_flip=False,
    rescale=1./255,
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

In [5]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

Found 8036 images belonging to 26 classes.
Found 1716 images belonging to 26 classes.


In [ ]:
# Membuat model MobileNet
def build_model(hp):
    base_model = MobileNet(
        input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False  # freeze layer untuk transfer learning

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(hp.Float('dropout', 0.2, 0.5, step=0.1))(x)
    x = Dense(hp.Int('dense_units', 64, 256, step=64), activation='relu')(x)
    outputs = Dense(NUM_CLASSES, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=outputs)

    model.compile(
        optimizer=Adam(
            learning_rate=hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])
        ),
        loss='categorical_crossentropy',
        metrics=['accuracy',
                 Precision(name='precision'),
                 Recall(name='recall')]
    )

    return model

In [ ]:
# Optimalisasi Hyperparameter dengan Random Search

tuner = RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=28,
    executions_per_trial=1,
    directory='tuning_mobilenet',
    project_name='mobilenet_random_search'
)

tuner.search(
    train_generator,
    validation_data=val_generator,
    epochs=4
)


Trial 28 Complete [00h 07m 19s]
val_accuracy: 0.7360140085220337

Best val_accuracy So Far: 0.8706293702125549
Total elapsed time: 03h 27m 33s


In [ ]:
# Hasil Parameter Tuning terbaik
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print("Best Hyperparameters:")
print("Learning rate:", best_hps.get('learning_rate'))
print("Dense units:", best_hps.get('dense_units'))
print("Dropout:", best_hps.get('dropout'))


Best Hyperparameters:
Learning rate: 0.001
Dense units: 192
Dropout: 0.4


In [11]:
import pandas as pd

# Ambil semua trial dari tuner
trials = tuner.oracle.trials

# Buat list hasil tuning
results = []
for trial_id, trial in trials.items():
    row = trial.hyperparameters.values.copy()  # ambil kombinasi hyperparam
    row['trial_id'] = trial_id
    row['val_accuracy'] = trial.score          # ambil val_accuracy
    results.append(row)

# Buat DataFrame
df_results = pd.DataFrame(results)

# Urutkan berdasarkan val_accuracy tertinggi
df_results = df_results.sort_values(by='val_accuracy', ascending=False)

# Tampilkan sebagai tabel
print("=== Tabel Hasil Tuning ===")
print(df_results)


=== Tabel Hasil Tuning ===
    dropout  dense_units  learning_rate trial_id  val_accuracy
20      0.4          192         0.0010       20      0.870629
23      0.3          256         0.0010       23      0.870047
6       0.3          192         0.0010       06      0.851981
9       0.4          256         0.0010       09      0.849650
14      0.3           64         0.0010       14      0.843240
13      0.2           64         0.0010       13      0.842075
25      0.4          128         0.0010       25      0.840909
0       0.2          128         0.0100       00      0.791958
10      0.3          256         0.0001       10      0.760490
17      0.3          256         0.0100       17      0.750583
27      0.2          256         0.0100       27      0.736014
16      0.2          256         0.0001       16      0.734266
4       0.4          256         0.0001       04      0.731352
5       0.4          192         0.0100       05      0.729021
2       0.3          128    

In [13]:
# Simpan ke dalam file CSV
df_results.to_csv("hasil_tuning_mobilenet.csv", index=False)

print("Hasil tuning telah disimpan ke dalam file 'hasil_tuning.csv'")

Hasil tuning telah disimpan ke dalam file 'hasil_tuning.csv'


In [1]:
import pandas as pd

# Membaca file hasil tuning dari CSV
df = pd.read_csv("hasil_tuning_mobilenet.csv")

# Menampilkan 5 baris pertama dari data
print("=== 5 Baris Pertama dari Hasil Tuning ===")
print(df.head(30))


=== 5 Baris Pertama dari Hasil Tuning ===
    dropout  dense_units  learning_rate  trial_id  val_accuracy
0       0.4          192         0.0010        20      0.870629
1       0.3          256         0.0010        23      0.870047
2       0.3          192         0.0010         6      0.851981
3       0.4          256         0.0010         9      0.849650
4       0.3           64         0.0010        14      0.843240
5       0.2           64         0.0010        13      0.842075
6       0.4          128         0.0010        25      0.840909
7       0.2          128         0.0100         0      0.791958
8       0.3          256         0.0001        10      0.760490
9       0.3          256         0.0100        17      0.750583
10      0.2          256         0.0100        27      0.736014
11      0.2          256         0.0001        16      0.734266
12      0.4          256         0.0001         4      0.731352
13      0.4          192         0.0100         5      0.72902